In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
PIPELINE_NAME = "Gold Customer 360"

SOURCE_TABLE_1 = SILVER_CUSTOMERS
SOURCE_TABLE_2 = GOLD_FACT_SALES

TARGET_TABLE = GOLD_CUSTOMER_360

RUN_ID = generate_run_id()
START_TIME = datetime.now()

In [0]:
print("GOLD CUSTOMER 360 PIPELINE")

print(f"Pipeline : {PIPELINE_NAME}")
print(f"Run ID : {RUN_ID}")
print(f"Target : {TARGET_TABLE}")

GOLD CUSTOMER 360 PIPELINE
Pipeline : Gold Customer 360
Run ID : 154013a0-b8ba-46d1-820f-ea2ff531be5e
Target : retailmart.gold.customer_360


In [0]:
customers_df = spark.table(SOURCE_TABLE_1)
fact_sales_df = spark.table(SOURCE_TABLE_2)

In [0]:
# Dataset Profile
print(f"Customers : {customers_df.count()}")
print(f"Fact Sales Records : {fact_sales_df.count()}")

customers_df.printSchema()
fact_sales_df.printSchema()

Customers : 15000
Fact Sales Records : 86328
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- pipeline_name: string (nullable = true)
 |-- run_id: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_month: string (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- delivery_duration_days: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = 

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {TARGET_TABLE} AS
WITH customer_metrics AS (

    SELECT

        customer_id,

        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(order_item_id) AS total_items_purchased,
        SUM(total_item_value) AS total_spent,
        AVG(total_item_value) AS average_item_value,
        MIN(order_purchase_timestamp) AS first_purchase,
        MAX(order_purchase_timestamp) AS last_purchase,

        DATEDIFF(
            MAX(order_purchase_timestamp),
            MIN(order_purchase_timestamp)
        ) AS customer_lifetime_days

    FROM {GOLD_FACT_SALES}
    GROUP BY customer_id
)

SELECT

    c.customer_id,
    c.customer_city,
    c.customer_state,

    COALESCE(m.total_orders,0) AS total_orders,
    COALESCE(m.total_items_purchased,0) AS total_items_purchased,
    COALESCE(m.total_spent,0) AS total_spent,
    COALESCE(m.average_item_value,0) AS average_item_value,

    m.first_purchase,
    m.last_purchase,

    COALESCE(m.customer_lifetime_days,0) AS customer_lifetime_days

FROM {SILVER_CUSTOMERS} c

LEFT JOIN customer_metrics m

ON c.customer_id = m.customer_id
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
customer_360_df = spark.table(TARGET_TABLE)
display(customer_360_df.limit(10))

customer_id,customer_city,customer_state,total_orders,total_items_purchased,total_spent,average_item_value,first_purchase,last_purchase,customer_lifetime_days
CUST_000018,Manaus,AM,7,21,26018.590000000004,1238.9804761904763,2021-04-02T21:19:00.000Z,2023-06-20T10:16:00.000Z,809
CUST_000036,Manaus,AM,3,8,10247.5,1280.9375,2021-03-02T03:32:00.000Z,2022-04-21T07:18:00.000Z,415
CUST_000066,Campo Grande,MS,4,4,4156.87,1039.2175,2021-06-09T23:00:00.000Z,2022-06-14T08:46:00.000Z,370
CUST_000089,Joao Pessoa,PB,5,15,21994.440000000002,1466.296,2021-04-10T06:34:00.000Z,2023-04-21T19:22:00.000Z,741
CUST_000095,Teresina,PI,1,1,1036.53,1036.53,2022-08-08T15:08:00.000Z,2022-08-08T15:08:00.000Z,0
CUST_000105,Recife,PE,4,6,11380.58,1896.7633333333333,2021-01-17T12:26:00.000Z,2022-07-30T07:39:00.000Z,559
CUST_000117,Manaus,AM,0,0,0.0,0.0,null,null,0
CUST_000119,Teresina,PI,8,11,13052.109999999999,1186.5554545454545,2021-02-01T12:21:00.000Z,2023-05-31T09:07:00.000Z,849
CUST_000136,Joao Pessoa,PB,1,3,3487.1,1162.3666666666666,2021-06-08T11:29:00.000Z,2021-06-08T11:29:00.000Z,0
CUST_000159,Porto Velho,RO,2,2,2271.27,1135.635,2021-11-11T15:22:00.000Z,2022-06-21T01:37:00.000Z,222


In [0]:
rows_written = customer_360_df.count()
unique_customers = customer_360_df.select("customer_id").distinct().count()

customers_without_orders = customer_360_df.filter(
    "total_orders = 0"
).count()

duplicate_customers = (
    customer_360_df.groupBy("customer_id")
    .count()
    .filter("count > 1")
    .count()
)

print(f"Rows Written : {rows_written}")
print(f"Unique Customers : {unique_customers}")
print(f"Customers Without Orders : {customers_without_orders}")
print(f"Duplicate Customers : {duplicate_customers}")

assert duplicate_customers == 0
assert rows_written == unique_customers

Rows Written : 15000
Unique Customers : 15000
Customers Without Orders : 519
Duplicate Customers : 0


In [0]:
source = f"{SOURCE_TABLE_1}, {SOURCE_TABLE_2}"
rows_read = customers_df.count() + fact_sales_df.count()
bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=source,
    target=TARGET_TABLE,
    rows_read=rows_read,
    rows_written=rows_written,
    duplicate_count=0,
    start_time=START_TIME,
    status="SUCCESS"
)

LOAD REPORT
Pipeline        : Gold Customer 360
Run ID          : 154013a0-b8ba-46d1-820f-ea2ff531be5e
Source          : retailmart.silver.customers, retailmart.gold.fact_sales
Target          : retailmart.gold.customer_360
Rows Read       : 101328
Rows Written    : 15000
Duplicate Rows  : 0
Start Time      : 2026-07-19 06:32:14.910484
End Time        : 2026-07-19 06:32:58.957794
Duration (sec)  : 44.05
Status          : SUCCESS


#Engineering Observations
- Customer-level metrics were generated by aggregating transactional data from the Gold Fact Sales table, ensuring that each customer is represented by a single consolidated record.
- A LEFT JOIN was performed between the Silver Customers table and the aggregated customer metrics to retain all registered customers, including those who have not placed any orders.
- Key customer KPIs such as total orders, total items purchased, total spending, average item value, first purchase date, last purchase date, and customer lifetime were calculated to provide a comprehensive customer profile.
- The COALESCE() function was used to replace NULL values with default values (0), ensuring consistency for customers without purchase history.
- The Gold Customer 360 table serves as a reusable analytics layer and acts as the foundation for downstream customer-focused datasets such as Customer Segmentation and Above Average Customers, reducing redundant transformations across the pipeline.